# IA102 — Notebook bonus — Les Transformers et le mécanisme d'attention

**Étude autonome, environ 30 minutes** · Prolongement de l'objectif O1. Inspiré de la partie 4 du cours de l'école de printemps 2026.

Les CNN excellent sur les images. Pour le **texte**, et aujourd'hui pour presque tout (GPT, BERT, LLaMA, modèles de vision), l'architecture dominante est le **Transformer** (Vaswani et al., 2017, *Attention Is All You Need*). Son ingrédient central : l'**attention**.

### Objectifs
- Transformer un texte en tenseurs : **tokenisation** et **plongements** (*embeddings*).
- Comprendre l'**encodage positionnel**.
- Programmer l'**auto-attention** pas à pas.
- Connaître l'attention **multi-têtes** et les modèles **préentraînés**.

In [ ]:
# Outil de questions de vérification (exécutez cette cellule sans la modifier)
import ipywidgets as widgets
from IPython.display import display, Markdown

def quiz(question, options, bonne, explication):
    """Affiche une question à choix multiples avec rétroaction immédiate."""
    choix = widgets.RadioButtons(options=options, value=None,
                                 layout={"width": "max-content"})
    bouton = widgets.Button(description="Vérifier", button_style="primary")
    sortie = widgets.Output()

    def verifier(_):
        with sortie:
            sortie.clear_output()
            if choix.value is None:
                display(Markdown("Veuillez choisir une réponse."))
            elif choix.value == options[bonne]:
                display(Markdown(f"**Bonne réponse.** {explication}"))
            else:
                display(Markdown("**Réponse incorrecte.** Relisez la section précédente et essayez de nouveau."))

    bouton.on_click(verifier)
    display(Markdown(f"**Question de vérification.** {question}"), choix, bouton, sortie)

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
plt.rcParams.update({"figure.dpi": 110})
torch.manual_seed(7)

## 1. Du texte aux vecteurs

Un réseau ne manipule que des nombres. Deux étapes :
1. **Tokeniser** : découper le texte en unités (*tokens*) et donner un numéro à chacune.
2. **Plonger** (*embed*) : associer à chaque numéro un **vecteur** appris. Des mots de sens proche finissent avec des vecteurs proches.

Les vrais modèles découpent en **sous-mots** (« ambulancière » → « ambul », « ancière »), ce qui permet de traiter des mots jamais vus. Ici, on découpe simplement par mot.

In [ ]:
phrase = "l ambulance arrive vite car le patient est grave"
mots = phrase.split()
vocabulaire = {mot: i for i, mot in enumerate(sorted(set(mots)))}
ids = torch.tensor([vocabulaire[m] for m in mots])
print("Tokens :", mots)
print("Identifiants :", ids.tolist())

d_modele = 16                                             # taille des vecteurs (512 dans l'article original)
plongement = nn.Embedding(len(vocabulaire), d_modele)
vecteurs = plongement(ids)
print("Forme après plongement :", tuple(vecteurs.shape), "→ (nombre de tokens, d_modele)")

## 2. L'encodage positionnel : donner un ordre aux mots

L'attention compare tous les tokens **en parallèle** : sans aide, elle ne sait pas que « le patient » vient après « car ». On **ajoute** donc à chaque vecteur un code de position, fait de sinus et cosinus de fréquences différentes :

$$PE_{(pos,\,2i)} = \sin\!\left(\frac{pos}{10000^{2i/d}}\right) \qquad PE_{(pos,\,2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d}}\right)$$

Chaque position reçoit ainsi une « signature » unique, un peu comme les aiguilles d'une horloge qui tournent à des vitesses différentes.

In [ ]:
def encodage_positionnel(n_positions, d):
    pe = torch.zeros(n_positions, d)
    position = torch.arange(n_positions).unsqueeze(1).float()
    frequences = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
    pe[:, 0::2] = torch.sin(position * frequences)
    pe[:, 1::2] = torch.cos(position * frequences)
    return pe

pe = encodage_positionnel(50, 64)
plt.figure(figsize=(9, 3.2))
plt.imshow(pe.T, cmap="RdBu", aspect="auto")
plt.xlabel("Position dans la phrase"); plt.ylabel("Dimension du vecteur")
plt.title("Encodage positionnel : une signature unique par position"); plt.colorbar(); plt.show()

entree = vecteurs + encodage_positionnel(len(mots), d_modele)
print("Entrée du Transformer :", tuple(entree.shape))

## 3. L'auto-attention : chaque mot regarde les autres

**Intuition : une recherche « floue » dans un dictionnaire.** Dans un dictionnaire Python, une **requête** doit correspondre **exactement** à une **clé** pour obtenir sa **valeur**. L'attention fait la même chose, mais en douceur : chaque requête est comparée à **toutes** les clés, et on récupère un **mélange** des valeurs, pondéré par la similarité.

Pour chaque token, le réseau calcule trois vecteurs avec trois couches linéaires apprises :
- **Q (requête)** : « qu'est-ce que je cherche ? »
- **K (clé)** : « qu'est-ce que j'offre ? »
- **V (valeur)** : « quelle information je transmets ? »

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V$$

1. $Q K^\top$ : la similarité entre chaque requête et chaque clé (un tableau n × n) ;
2. $/\sqrt{d_k}$ : une mise à l'échelle pour garder des nombres raisonnables ;
3. **softmax** : chaque ligne devient une distribution de poids qui somme à 1 ;
4. $\times V$ : chaque token reçoit la moyenne pondérée des valeurs.

### Exercice B1 — Programmer l'attention

In [ ]:
def attention(Q, K, V):
    d_k = Q.shape[-1]
    scores = ...                     # À COMPLÉTER : Q @ K transposée, divisé par racine de d_k
    poids = ...                      # À COMPLÉTER : softmax sur la dernière dimension
    return poids @ V, poids

d_k = 8
projection_q, projection_k, projection_v = nn.Linear(d_modele, d_k), nn.Linear(d_modele, d_k), nn.Linear(d_modele, d_k)
with torch.no_grad():
    Q, K, V = projection_q(entree), projection_k(entree), projection_v(entree)
    sortie, poids = attention(Q, K, V)
    reference = F.scaled_dot_product_attention(Q, K, V)

print("Forme de la sortie :", tuple(sortie.shape))
print("Identique à la fonction de PyTorch :", torch.allclose(sortie, reference, atol=1e-6))
print("Chaque ligne de poids somme à 1 :", torch.allclose(poids.sum(-1), torch.ones(len(mots))))

<details>
<summary><b>Afficher la solution</b> (après avoir fait l'exercice)</summary>

```python
def attention(Q, K, V):
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    poids = torch.softmax(scores, dim=-1)
    return poids @ V, poids

d_k = 8
projection_q, projection_k, projection_v = nn.Linear(d_modele, d_k), nn.Linear(d_modele, d_k), nn.Linear(d_modele, d_k)
with torch.no_grad():
    Q, K, V = projection_q(entree), projection_k(entree), projection_v(entree)
    sortie, poids = attention(Q, K, V)
    reference = F.scaled_dot_product_attention(Q, K, V)

print("Forme de la sortie :", tuple(sortie.shape))
print("Identique à la fonction de PyTorch :", torch.allclose(sortie, reference, atol=1e-6))
print("Chaque ligne de poids somme à 1 :", torch.allclose(poids.sum(-1), torch.ones(len(mots))))
```

</details>

In [ ]:
plt.figure(figsize=(6.5, 5.5))
plt.imshow(poids, cmap="viridis")
plt.xticks(range(len(mots)), mots, rotation=45, ha="right"); plt.yticks(range(len(mots)), mots)
plt.xlabel("… regarde ce token (clé)"); plt.ylabel("Ce token (requête)…")
plt.title("Poids d'attention (réseau non entraîné)"); plt.colorbar(); plt.show()

Ici, les projections sont **aléatoires** : les poids n'ont pas encore de sens. Après entraînement sur beaucoup de texte, « grave » apprendrait par exemple à regarder fortement « patient », et « arrive » à regarder « ambulance ». Le modèle apprend **seul** quelles relations sont utiles.

## 4. L'attention multi-têtes

Une seule tête d'attention ne capte qu'un type de relation. On en utilise **plusieurs en parallèle** (8, 16, 96…) : une tête peut suivre la grammaire, une autre le sens, une autre la coréférence. Leurs sorties sont concaténées, puis reprojetées.

In [ ]:
multi = nn.MultiheadAttention(embed_dim=d_modele, num_heads=4, batch_first=True)
lot = entree.unsqueeze(0)                                  # (lot = 1, tokens, d_modele)
with torch.no_grad():
    sortie_multi, poids_multi = multi(lot, lot, lot, average_attn_weights=False)
print("Sortie :", tuple(sortie_multi.shape), "| poids :", tuple(poids_multi.shape), "→ (lot, têtes, tokens, tokens)")

fig, axes = plt.subplots(1, 4, figsize=(14, 3.3))
for t, ax in enumerate(axes):
    ax.imshow(poids_multi[0, t], cmap="viridis"); ax.set_title(f"Tête {t + 1}"); ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("Chaque tête apprend ses propres relations"); plt.show()

## 5. Un bloc Transformer complet

Un bloc empile : attention multi-têtes → addition résiduelle et normalisation → petit réseau dense → addition résiduelle et normalisation. Un grand modèle de langage empile des **dizaines** de ces blocs.

In [ ]:
bloc = nn.TransformerEncoderLayer(d_model=d_modele, nhead=4, dim_feedforward=64, batch_first=True)
encodeur = nn.TransformerEncoder(bloc, num_layers=6)
with torch.no_grad():
    print("Sortie de 6 blocs :", tuple(encodeur(lot).shape))
print("Paramètres :", sum(p.numel() for p in encodeur.parameters()))

## 6. En pratique : les modèles préentraînés

On n'entraîne presque jamais un Transformer à partir de zéro : on part d'un modèle **préentraîné** (le transfert d'apprentissage du notebook 4, appliqué au texte), disponible par exemple sur **Hugging Face** :

| Famille | Exemples | Usage typique |
| --- | --- | --- |
| Encodeurs | BERT, CamemBERT (français) | Classification de textes, extraction d'entités |
| Décodeurs | GPT, LLaMA, Mistral | Génération de texte, assistants |
| Encodeurs-décodeurs | T5, BART | Traduction, résumé |

La cellule suivante demande la bibliothèque `transformers` et un accès à Internet (ou un modèle téléchargé à l'avance).

In [ ]:
try:
    from transformers import pipeline
    classifieur = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment")
    for texte in ["Cet atelier était vraiment excellent !", "Je n'ai rien compris, c'était trop rapide."]:
        print(texte, "→", classifieur(texte)[0])
except Exception as erreur:
    print("Exemple Hugging Face non exécuté :", type(erreur).__name__)
    print("Installez la bibliothèque (pip install transformers) sur une machine connectée à Internet pour exécuter cet exemple.")

In [ ]:
quiz("Dans l'attention, à quoi sert le softmax ?",
     ["À transformer les similarités en poids positifs qui somment à 1", "À réduire la taille des vecteurs", "À ajouter la position des mots"],
     0, "Chaque requête répartit ainsi « 100 % de son attention » entre tous les tokens.")

In [ ]:
quiz("Pourquoi ajoute-t-on un encodage positionnel ?",
     ["Pour accélérer le calcul", "Parce que l'attention, à elle seule, ignore l'ordre des tokens", "Pour réduire le vocabulaire"],
     1, "L'attention traite les tokens comme un ensemble ; l'encodage positionnel réintroduit l'ordre.")

## À retenir
- Texte → **tokens** → **plongements** (vecteurs appris) + **encodage positionnel**.
- **Auto-attention** : softmax(QKᵀ / √d) V — chaque token fait une moyenne pondérée de tous les autres.
- **Multi-têtes** : plusieurs attentions en parallèle, chacune avec ses propres relations.
- En pratique, on **adapte un modèle préentraîné** plutôt que d'entraîner de zéro.